# Module C — Retrieval Models

1. Model 1: Lexical Retrieval (BM25)
2. Model 2: Fuzzy/Transliteration Matching
3. Model 3: Semantic Matching
4. Model 4: Hybrid Ranking

In [12]:
import feedparser
import json
from tqdm import tqdm
import os
from bs4 import BeautifulSoup
import html
import re
import pickle
from rank_bm25 import BM25Okapi

## 3. Configuration

### 3.1 Path

In [13]:
EN_PATH = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data\document_en_clean.json"
BN_PATH = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data\document_bn_clean.json"

EN_INDEX_OUT='bm25_en.pkl'
BN_INDEX_OUT='bm25_bn.pkl'

### 3.2 Tokenizer

#### 3.2.1 English Tokenizer

In [14]:
def tokenize_en(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

#### 3.2.1 Bangla Tokenizer

In [15]:
def tokenize_bn(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

### 3.3 Build and Save English BM25 index

In [16]:
with open(EN_PATH, 'r', encoding='utf-8') as f:
    docs_en = json.load(f)

corpus_end=[]
doc_ids_en=[]

for doc in docs_en:
    title = doc.get("title", "")
    body = doc.get("body", "")

    full_text = (title + " " + body).strip()

    corpus_end.append(tokenize_en(full_text))
    doc_ids_en.append(doc.get("doc_id"," "))

bm25_en = BM25Okapi(corpus_end)

with open(EN_INDEX_OUT, 'wb') as f:
    pickle.dump({"bm25": bm25_en, "doc_ids": doc_ids_en, "docs": docs_en}, f)

print(f"English docs indexing:" ,len(doc_ids_en))
print(f"BM25 English index saved to:" ,EN_INDEX_OUT)

English docs indexing: 150
BM25 English index saved to: bm25_en.pkl


### 3.4 Build and Save Bangla BM25 index

In [17]:
with open(BN_PATH, 'r', encoding='utf-8') as f:
    docs_bn = json.load(f)

corpus_end=[]
doc_ids_bn=[]

for doc in docs_bn:
    title = doc.get("title", "")
    body = doc.get("body", "")

    full_text = (title + " " + body).strip()

    corpus_end.append(tokenize_bn(full_text))
    doc_ids_bn.append(doc.get("doc_id"," "))

bm25_bn = BM25Okapi(corpus_end)

with open(BN_INDEX_OUT, 'wb') as f:
    pickle.dump({"bm25": bm25_bn, "doc_ids": doc_ids_bn, "docs": docs_bn}, f)

print(f"Bengali docs indexing:" ,len(doc_ids_bn))
print(f"BM25 Bengali index saved to:" ,BN_INDEX_OUT)

Bengali docs indexing: 146
BM25 Bengali index saved to: bm25_bn.pkl


### 3.5 Load Indexes and Search Function

#### 3.5.1 Load Indexes

In [18]:
def load_index(path):
    with open(path, "rb") as f:
        return pickle.load(f)

en_pack = load_index(EN_INDEX_OUT)
bn_pack = load_index(BN_INDEX_OUT)

bm25_en = en_pack["bm25"]
doc_ids_en = en_pack["doc_ids"]
docs_en = en_pack["docs"]

bm25_bn = bn_pack["bm25"]
doc_ids_bn = bn_pack["doc_ids"]
docs_bn = bn_pack["docs"]

#### 3.5.2 English Search Function

In [19]:
def search_en(query, top_k=5):
    q = tokenize_en(query)
    scores = bm25_en.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_en[i], "score": float(scores[i]), "title": docs_en[i].get("title",""), "url": docs_en[i].get("url","")} for i in idx]


#### 3.5.3 Bangla Search Function

In [20]:
def search_bn(query, top_k=5):
    q = tokenize_bn(query)
    scores = bm25_bn.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_bn[i], "score": float(scores[i]), "title": docs_bn[i].get("title",""), "url": docs_bn[i].get("url","")} for i in idx]

### 3.6 Test

#### 3.6.1 Test English Query

In [21]:
for r in search_en("Bangladesh cricket", top_k=5):
    print(r["score"], "|", r["doc_id"], "|", r["title"])
    print("  ", r["url"])

6.416134376204756 | en_000175 | the year 2025 will be remembered less for what bangladesh cricket achieved on the field and more for the turbulence that engulfed it off it..
   https://www.thedailystar.net/sports/cricket/news/year-turmoil-bangladesh-cricket-4066656
5.887880086428844 | en_000174 | cricket australia chief todd greenberg said saturday that short tests were bad for business as some of the biggest names in the game attacked the state of the melbourne cricket ground pitch.
   https://www.thedailystar.net/sports/cricket/news/cricket-australia-boss-says-short-tests-bad-business-after-mcg-carnage-4066956
5.647109687728009 | en_000173 | chattogram royals captain mahedi hasan admitted he had never experienced such instability in a franchise in the bangladesh premier league (bpl) after leading his side to a 65-run win over noakhali express at the sylhet international cricket stadium on friday.
   https://www.thedailystar.net/sports/sports-special/bpl-2026/news/better-happened-the-

#### 3.6.2 Test Bangla Query

In [22]:
for r in search_bn("বাংলাদেশ ক্রিকেট", top_k=5):
    print(r["score"], "|", r["doc_id"], "|", r["title"])
    print("  ", r["url"])

7.399980740492559 | bn_000082 | ৪ বছর পর বিপিএলে শান্তর সেঞ্চুরি
   https://www.risingbd.com/sports/news/633526
4.819759886943764 | bn_000045 | চট্টগ্রামের কাছে পাত্তা পেল না নবাগত নোয়াখালী এক্সপ্রেস
   https://www.risingbd.com/sports/news/633563
4.793073593425571 | bn_000054 | বেগের ব্যাটে চ্যালেঞ্জিং সংগ্রহ পেল চট্টগ্রাম
   https://www.risingbd.com/sports/news/633554
4.149732889662082 | bn_000145 | টস জিতে শান্তর রাজশাহীকে ব্যাটিংয়ে পাঠালো মিঠুনের ঢাকা
   https://www.jagonews24.com/sports/cricket/1079254
3.3866996719499656 | bn_000168 | ৭ রানে ৮ উইকেট, আন্তর্জাতিক টি-টোয়েন্টিতে বিশ্বরেকর্ড
   https://www.jagonews24.com/sports/cricket/1079231
